# Data Exploration

**AirQuito** — notebook 01 of the pipeline.

Establishes what the REMMAQ data contains, before any pipeline table is designed: the
historical exports from `datosambiente.quito.gob.ec`, the live feed behind
`aireambiente.quito.gob.ec`, and the Secretaría's own QA/QC framework. The questions are the
ones Silver and Gold will have to answer in code: which stations exist and when, what the
timestamps mean, what a valid value looks like, how the live feed and the history join, and
what the product can honestly show.

Every section follows the same shape: a line on *why* the check is run, the code, then a
**Finding** block recording the result. The findings are the deliverable — this notebook is
the data-quality record for the project, not a scratchpad. The Summary at the end turns them
into the objectives for Stage 1 and the rules for Silver.

| Input | Where it came from |
|---|---|
| `remmaq/PM2.5.xlsx`, `NO2.xlsx`, `O3.xlsx` | `datosambiente.quito.gob.ec`, one `.rar` per parameter, extracted by hand (2026-10-16) |
| `remmaq/Marco_QAQC_REMMAQ.pdf` | Same portal: the network's validation protocol, 5 pages |
| `probe/<run>/secretaria_get_sites_info.json` | One call to the live feed by `scripts/probe_sources.py` (2026-10-16 23:41 UTC) |
| `sink/aq.measurements.raw/*.jsonl` | Output of `producers/remmaq_producer.py` with the file sink (optional) |

---

## 0. Configuration

The notebook runs unchanged on Databricks (where `spark` and `.display()` already exist and the
data sits in a Unity Catalog volume) and on plain PySpark (`AIRQUITO_DATA_ROOT=/path/to/data/local`),
so the findings can be re-verified anywhere the files are.

Two things are deliberately set here and nowhere else: the timezone (Quito is UTC-5 all year,
no daylight saving) and the station reference table, which is the same one the producers use.

In [1]:
import os, sys, json, glob, time
from pathlib import Path
import pandas as pd
from pyspark.sql import functions as F, Window as W, DataFrame

# Data root. Databricks volume by default; override with AIRQUITO_DATA_ROOT elsewhere.
BASE = Path(os.environ.get("AIRQUITO_DATA_ROOT", "/Volumes/workspace/default/airquito"))
REMMAQ = BASE / "remmaq"
CACHE = REMMAQ / "parquet"          # xlsx converted once, see section 2

# Outside Databricks there is no ambient `spark` and DataFrame has no .display():
# create a local session and make .display() an alias of .show().
try:
    spark  # noqa: F821  (Databricks injects it)
except NameError:
    from pyspark.sql import SparkSession
    spark = (SparkSession.builder.master("local[*]").appName("01_exploring")
             .config("spark.sql.shuffle.partitions", "8")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")
if not hasattr(DataFrame, "display"):
    DataFrame.display = lambda self, n=60: self.show(n, truncate=False)

QUITO_TZ = "America/Guayaquil"      # UTC-5, no DST

# Station reference: the producers' table when the repo is importable (local run or a
# Databricks Git folder), otherwise the same rows inline. Three identifiers per station must
# join: the feed's GUID (message key), the feed's display name, and the xlsx column header.
try:
    sys.path.insert(0, str(Path.cwd().parent))
    from producers.stations import STATIONS
    stations = [(s.station_id, s.name, s.history_column, s.latitude, s.longitude) for s in STATIONS]
    print("stations from producers/stations.py")
except ImportError:
    stations = [
        ("f1f5d7e6-09f3-4966-ad15-f6ad47b89973", "Belisario", "BELISARIO", -0.184719, -78.495986),
        ("35e166f8-0d20-4955-84ec-1180d8193d03", "Carapungo", "CARAPUNGO", -0.095472, -78.449809),
        ("1f090827-f3b4-4c46-9794-af4e2d46c126", "Centro", "CENTRO", -0.221393, -78.514005),
        ("da0e2ba0-92ab-4703-80b3-825144c34659", "Cotocollao", "COTOCOLLAO", -0.107777, -78.497222),
        ("f1634ea4-d0d0-4b18-b00a-e25e7c06c8be", "El Camal", "EL CAMAL", -0.25, -78.51),
        ("dbf143fe-e4dc-4875-bdec-70cca4dde273", "Guamani", "GUAMANI", -0.333949, -78.553416),
        ("4cf14787-3ed1-4194-97b6-7f038d76f6d0", "LosChillos", "LOS CHILLOS", -0.297062, -78.455248),
        ("96c776fd-b679-40e6-ae95-d657e05e3a84", "SanAntonio", "SAN ANTONIO", -0.009222, -78.448001),
        ("54c686c1-ad66-4c69-b6dc-54dc6ac9eecd", "Tumbaco", "TUMBACO", -0.214933, -78.403253),
    ]
    print("stations inline (producers package not importable)")
stations_df = spark.createDataFrame(stations, ["station_id", "feed_name", "station", "latitude", "longitude"])

# The three history files and the feed layer each one corresponds to.
FILES = {"PM2.5_ug": "PM2.5.xlsx", "NO2_ug": "NO2.xlsx", "OZONO_ug": "O3.xlsx"}
print("data root:", BASE, "| exists:", BASE.exists())

26/10/17 03:07:20 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/10/17 03:07:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/17 03:07:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


stations from producers/stations.py


data root: /workspace/quito-air-conditions/data/local | exists: True


---

## 1. Physical layout

Walk the folder before reading anything, and open only the header row of each workbook:
sheet names, column headers and row counts are visible without loading 10 MB of cells.

In [2]:
for sub in ["remmaq", "probe", "sink"]:
    root = BASE / sub
    if not root.exists():
        print(f"{sub}/  (absent)"); continue
    for path in sorted(root.rglob("*")):
        if path.is_file() and "parquet" not in str(path):
            print(f"{path.relative_to(BASE)!s:<60}{path.stat().st_size/1e6:8.1f} MB")

remmaq/Marco_QAQC_REMMAQ.pdf                                     0.7 MB
remmaq/NO2.xlsx                                                 10.3 MB
remmaq/O3.xlsx                                                  12.2 MB
remmaq/PM2.5.xlsx                                               10.2 MB
probe/20261016T223521Z/airgradient_quito_subset.json             0.0 MB
probe/20261016T223521Z/airgradient_world_current.json            0.3 MB
probe/20261016T223521Z/openaq_latest_2834723.json                0.0 MB
probe/20261016T223521Z/openaq_latest_2838228.json                0.0 MB
probe/20261016T223521Z/openaq_latest_42641.json                  0.0 MB
probe/20261016T223521Z/openaq_latest_42642.json                  0.0 MB
probe/20261016T223521Z/openaq_latest_42643.json                  0.0 MB
probe/20261016T223521Z/openaq_latest_42644.json                  0.0 MB
probe/20261016T223521Z/openaq_latest_42645.json                  0.0 MB
probe/20261016T223521Z/openaq_latest_42646.json                 

In [3]:
import openpyxl

for param, name in FILES.items():
    wb = openpyxl.load_workbook(REMMAQ / name, read_only=True)
    for sheet in wb.sheetnames:
        ws = wb[sheet]
        rows = ws.iter_rows(values_only=True)
        header, first = next(rows), next(rows)
        print(f"{name:<11} sheet={sheet!r:<9} rows={ws.max_row:<8} cols={ws.max_column}")
        print(f"{'':11} header: {header}")
        print(f"{'':11} first : {first}")

PM2.5.xlsx  sheet='LIMPIO'  rows=192970   cols=10
            header: ('FECHA', 'BELISARIO', 'CARAPUNGO', 'CENTRO', 'COTOCOLLAO', 'EL CAMAL', 'GUAMANI', 'LOS CHILLOS', 'SAN ANTONIO', 'TUMBACO')
            first : (datetime.datetime(2004, 8, 26, 15, 0), None, None, 16, None, None, None, None, None, None)
NO2.xlsx    sheet='LIMPIO'  rows=198506   cols=9
            header: ('FECHA', 'BELISARIO', 'CARAPUNGO', 'CENTRO', 'COTOCOLLAO', 'EL CAMAL', 'GUAMANI', 'LOS CHILLOS', 'TUMBACO')
            first : (datetime.datetime(2004, 1, 1, 0, 0), None, None, 121.37, None, 108.73, None, None, None)
O3.xlsx     sheet='LIMPIO'  rows=198510   cols=10
            header: ('Fecha', 'BELISARIO', 'CARAPUNGO', 'CENTRO', 'COTOCOLLAO', 'EL CAMAL', 'GUAMANI', 'LOS CHILLOS', 'SAN ANTONIO', 'TUMBACO')
            first : (datetime.datetime(2004, 1, 1, 0, 0), 42.14, None, None, None, 90.85, None, 27.86, None, 8.54)


> **Finding**
>
> - One workbook per parameter, one sheet named **`LIMPIO`** ("clean"): this is the Secretaría's
>   *validated* series, the output of the QA/QC protocol in the PDF, not raw instrument data.
> - **Wide format**: a timestamp column followed by one column per station. The pipeline's
>   long format (one row per station, parameter, hour) has to be produced by unpivoting.
> - Schema drift between files of the same publisher, already:
>   - `O3.xlsx` names the timestamp column **`Fecha`**; the other two say `FECHA`.
>   - `NO2.xlsx` has **no `SAN ANTONIO` column** (8 stations); PM2.5 and O3 have 9.
>   - `PM2.5.xlsx` starts 2004-08-26, the others 2004-01-01.
> - The first data rows are mostly empty: stations were commissioned over years (section 5).
>
> Consequence: the loader normalises headers (`strip().upper()`) and must not assume a fixed
> station list per file. Belisario, absent from OpenAQ, is present here from 2004.

---

## 2. Load once, then Spark

Reading xlsx is slow (about 20 s per file through openpyxl) and Spark has no native Excel
reader, so each workbook is read once with pandas, unpivoted to long format and written to
parquet under `remmaq/parquet/<parameter>/`. Every later cell reads parquet. Delete the
folder to force a rebuild.

One trap is handled here on purpose: pandas represents missing cells as `NaN`, and Spark
treats `NaN` as a *value*, not a null. Without the conversion below, `count(value)` reports
100 % coverage everywhere.

In [4]:
if not CACHE.exists():
    for param, name in FILES.items():
        t0 = time.time()
        pdf = pd.read_excel(REMMAQ / name, sheet_name="LIMPIO")
        pdf.columns = [c.strip().upper() for c in pdf.columns]
        pdf = pdf.rename(columns={"FECHA": "ts"})
        long = pdf.melt(id_vars="ts", var_name="station", value_name="value")
        long["parameter"] = param
        sdf = spark.createDataFrame(long).withColumn(
            "value", F.when(F.isnan("value"), None).otherwise(F.col("value")))
        sdf.write.mode("overwrite").parquet(str(CACHE / param))
        print(f"{param:<9} {len(pdf):>7} rows x {len(pdf.columns)-1} stations -> parquet in {time.time()-t0:.0f}s")
else:
    print("parquet cache present, skipping conversion")

raw = spark.read.option("basePath", str(CACHE)).parquet(str(CACHE / "*"))
raw.printSchema()
(raw.groupBy("parameter")
    .agg(F.count("*").alias("rows"), F.count("value").alias("non_null"),
         F.round(F.count("value") / F.count("*") * 100, 1).alias("non_null_pct"),
         F.countDistinct("station").alias("stations"), F.min("ts"), F.max("ts"))
    .orderBy("parameter").display())

parquet cache present, skipping conversion


root
 |-- ts: timestamp (nullable = true)
 |-- station: string (nullable = true)
 |-- value: double (nullable = true)
 |-- parameter: string (nullable = true)



+---------+-------+--------+------------+--------+-------------------+-----------------------+
|parameter|rows   |non_null|non_null_pct|stations|min(ts)            |max(ts)                |
+---------+-------+--------+------------+--------+-------------------+-----------------------+
|NO2_ug   |1588040|1176982 |74.1        |8       |2004-01-01 00:00:00|2026-08-31 23:00:07.465|
|OZONO_ug |1786581|1461324 |81.8        |9       |2004-01-01 00:00:00|2026-08-31 23:00:07.455|
|PM2.5_ug |1736721|1190105 |68.5        |9       |2004-08-26 15:00:00|2026-08-31 23:00:07.455|
+---------+-------+--------+------------+--------+-------------------+-----------------------+



> **Finding**
>
> | parameter | rows (station x hour) | non-null | stations | span |
> |---|---|---|---|---|
> | PM2.5 | 1 736 721 | 1 190 105 (68.5 %) | 9 | 2004-08-26 → 2026-08-31 |
> | NO2 | 1 588 040 | 1 176 982 (74.1 %) | 8 | 2004-01-01 → 2026-08-31 |
> | O3 | 1 786 581 | 1 461 324 (81.8 %) | 9 | 2004-01-01 → 2026-08-31 |
>
> **22 years of hourly data, 3.8 million validated values.** The export ends on
> **2026-08-31**; the live feed was captured on 2026-10-16. The six weeks between are in
> neither source yet — a backfill boundary to remember when the trend charts are built.
>
> A third of the station-hours are empty, but not at random: it is whole stations that did not
> exist yet or were offline (section 5), not scattered holes.

---

## 3. Timestamps

`max(ts)` above ends in `23:00:07.455`, which is not an hourly stamp. Measure the offset of
every timestamp from its nearest hour, by month, to see whether this is noise or a pattern —
the answer decides whether the loader can truncate or must round.

In [5]:
stamps = (raw.select("parameter", "ts").distinct()
             .withColumn("nearest", F.timestamp_seconds(F.round(F.unix_timestamp("ts") / 3600) * 3600))
             .withColumn("offset_s", F.col("ts").cast("double") - F.col("nearest").cast("double")))

(stamps.filter("offset_s != 0")
       .groupBy("parameter", F.date_format("nearest", "yyyy-MM").alias("month"))
       .agg(F.count("*").alias("stamps"), F.round(F.min("offset_s"), 3).alias("min_offset_s"),
            F.round(F.max("offset_s"), 3).alias("max_offset_s"))
       .filter("parameter = 'PM2.5_ug'").orderBy("month").display())

+---------+-------+------+------------+------------+
|parameter|month  |stamps|min_offset_s|max_offset_s|
+---------+-------+------+------------+------------+
|PM2.5_ug |2025-12|744   |-0.005      |-0.005      |
|PM2.5_ug |2026-01|744   |-0.005      |-0.005      |
|PM2.5_ug |2026-02|672   |-0.005      |-0.005      |
|PM2.5_ug |2026-03|744   |-0.005      |-0.005      |
|PM2.5_ug |2026-04|720   |-0.005      |-0.005      |
|PM2.5_ug |2026-05|744   |-0.005      |-0.005      |
|PM2.5_ug |2026-06|720   |-0.005      |-0.005      |
|PM2.5_ug |2026-07|744   |0.02        |3.735       |
|PM2.5_ug |2026-08|744   |3.74        |7.455       |
+---------+-------+------+------------+------------+



In [6]:
# Round to the nearest hour, then check that the result is a clean hourly grid:
# no duplicate (station, hour) and no missing hours between the first and the last.
hist = (raw.withColumn("ts_hour", F.timestamp_seconds(F.round(F.unix_timestamp("ts") / 3600) * 3600))
           .drop("ts"))

dups = hist.groupBy("parameter", "station", "ts_hour").count().filter("count > 1")
print("duplicate (parameter, station, hour) rows:", dups.count())

grid = (hist.groupBy("parameter")
            .agg(F.min("ts_hour").alias("first"), F.max("ts_hour").alias("last"),
                 F.countDistinct("ts_hour").alias("hours_present"))
            .withColumn("hours_expected", ((F.unix_timestamp("last") - F.unix_timestamp("first")) / 3600 + 1).cast("long"))
            .withColumn("hours_missing", F.col("hours_expected") - F.col("hours_present")))
grid.orderBy("parameter").display()

duplicate (parameter, station, hour) rows: 0


+---------+-------------------+-------------------+-------------+--------------+-------------+
|parameter|first              |last               |hours_present|hours_expected|hours_missing|
+---------+-------------------+-------------------+-------------+--------------+-------------+
|NO2_ug   |2004-01-01 00:00:00|2026-08-31 23:00:00|198505       |198696        |191          |
|OZONO_ug |2004-01-01 00:00:00|2026-08-31 23:00:00|198509       |198696        |187          |
|PM2.5_ug |2004-08-26 15:00:00|2026-08-31 23:00:00|192969       |192969        |0            |
+---------+-------------------+-------------------+-------------+--------------+-------------+



In [7]:
# Where are the missing hours? (rows absent from the export, as opposed to null values)
def missing_hours(param):
    first, last = grid.filter(F.col("parameter") == param).select("first", "last").first()
    n = int((last - first).total_seconds() // 3600) + 1
    full = spark.range(n).select((F.lit(first).cast("long") + F.col("id") * 3600).cast("timestamp").alias("ts_hour"))
    present = hist.filter(F.col("parameter") == param).select("ts_hour").distinct()
    return full.join(present, "ts_hour", "left_anti")

(missing_hours("OZONO_ug").groupBy(F.to_date("ts_hour").alias("day")).count()
                          .orderBy(F.desc("count"), "day").display(12))

+----------+-----+
|day       |count|
+----------+-----+
|2017-11-24|24   |
|2017-11-25|24   |
|2017-11-26|24   |
|2017-11-27|24   |
|2018-11-30|15   |
|2013-08-30|12   |
|2017-11-23|12   |
|2017-11-28|10   |
|2018-12-01|10   |
|2022-12-07|6    |
|2020-05-13|5    |
|2023-02-15|5    |
+----------+-----+
only showing top 12 rows



> **Finding — the export clock drifts, round, never truncate**
>
> Until November 2025 every stamp is exactly on the hour. From **2025-12-01** the stamps sit
> **5 ms before the hour** (`23:59:59.995` for the 00:00 reading) for seven months, then from
> July 2026 the offset turns positive and grows to **+7.5 s** by the end of August. This is a
> drifting clock in the export tool, not in the instruments: all three files show identical
> offsets for the same hours.
>
> Truncating to the hour (`date_trunc`) would have filed seven months of readings under the
> *previous* hour, an off-by-one that no row count would reveal. **Rounding to the nearest
> hour** yields zero duplicates and, for PM2.5, a complete grid of 192 969 hours.
>
> NO2 and O3 miss 191 and 187 *rows* respectively (hours absent from the file rather than
> empty cells), concentrated in a few multi-day stretches such as 2017-11-23 → 11-28. The
> loader must treat a missing row and a null value the same way: no observation.
>
> The live feed has no such problem: it stamps with epoch milliseconds (`/Date(1792188000000)/`),
> which the producer converts to UTC. But the same rule applies to the Silver merge of the two
> sources: **one row per (station, parameter, hour), hour rounded, in UTC.**

---

## 4. Timezone

The xlsx stamps carry no offset. The feed sends epoch milliseconds (UTC) *and* a local
string (`AqiTimeString: 2026-10-16T17:00:00` for epoch 22:00 UTC), so the feed is
unambiguous; the history is not. Physics settles it: ozone is produced by sunlight and peaks
around solar noon, traffic pollutants peak in the morning rush. If the hour-of-day averages
peak where they should in Quito local time, the stamps are local.

In [8]:
diurnal = (hist.filter("value is not null")
               .groupBy(F.hour("ts_hour").alias("hour_of_day"))
               .pivot("parameter", list(FILES))
               .agg(F.round(F.avg("value"), 1))
               .orderBy("hour_of_day"))
diurnal.display(24)
peaks = diurnal.toPandas().set_index("hour_of_day").idxmax()
print("hour of the daily maximum:", peaks.to_dict())

+-----------+--------+------+--------+
|hour_of_day|PM2.5_ug|NO2_ug|OZONO_ug|
+-----------+--------+------+--------+
|0          |15.4    |21.1  |15.1    |
|1          |14.8    |19.1  |15.6    |
|2          |13.5    |17.9  |15.8    |
|3          |13.1    |18.0  |15.0    |
|4          |13.5    |19.4  |13.2    |
|5          |15.1    |24.0  |10.0    |
|6          |21.4    |29.7  |7.8     |
|7          |24.6    |31.5  |12.0    |
|8          |22.6    |29.2  |21.4    |
|9          |19.7    |25.3  |32.7    |
|10         |18.2    |22.0  |43.2    |
|11         |17.2    |19.2  |51.1    |
|12         |16.0    |17.1  |54.0    |
|13         |15.3    |15.8  |51.6    |
|14         |15.4    |15.3  |47.2    |
|15         |15.1    |16.6  |42.0    |
|16         |15.1    |19.2  |36.1    |
|17         |15.7    |23.8  |28.4    |
|18         |16.5    |29.3  |20.1    |
|19         |16.2    |31.2  |15.8    |
|20         |16.2    |30.5  |13.8    |
|21         |15.6    |28.7  |13.2    |
|22         |14.5    |26.

hour of the daily maximum: {'PM2.5_ug': 7, 'NO2_ug': 7, 'OZONO_ug': 12}


> **Finding — history is in Quito local time (UTC-5)**
>
> | parameter | peak hour | physical reason |
> |---|---|---|
> | O3 | **12:00** | photochemical production, maximal at solar noon (Quito is on the equator at 78.5° W, solar noon ≈ 12:15) |
> | NO2 | **07:00** and 19:00 | morning and evening traffic |
> | PM2.5 | **07:00** | morning traffic plus the nocturnal inversion that has not yet broken |
>
> Had the stamps been UTC, every peak would appear five hours later. They do not.
>
> Consequence for Silver: `event_time = to_utc_timestamp(ts_hour, 'America/Guayaquil')` for the
> history; the feed is already UTC. Ecuador has no daylight saving, so the conversion is a
> constant five hours and never ambiguous.
>
> Open question, recorded not resolved: whether the hour label means the hour *ending* or
> *starting* at that time. The ozone peak at 12:00 rather than 13:00 slightly favours "hour
> starting", but the averaging window is an hour, so the product never needs finer than that.

---

## 5. Station coverage

Which stations measured what, and when. Coverage per year answers the first question a map
has to answer honestly: *which of these dots are alive.*

In [9]:
for param in FILES:
    print(f"=== {param}: % of hours with a value, per year ===")
    (hist.filter(F.col("parameter") == param)
         .groupBy(F.year("ts_hour").alias("year"))
         .pivot("station")
         .agg(F.round(F.count("value") / F.count("*") * 100))
         .orderBy("year").display(30))

=== PM2.5_ug: % of hours with a value, per year ===


+----+---------+---------+------+----------+--------+-------+-----------+-----------+-------+
|year|BELISARIO|CARAPUNGO|CENTRO|COTOCOLLAO|EL CAMAL|GUAMANI|LOS CHILLOS|SAN ANTONIO|TUMBACO|
+----+---------+---------+------+----------+--------+-------+-----------+-----------+-------+
|2004|91.0     |0.0      |96.0  |0.0       |96.0    |0.0    |0.0        |0.0        |0.0    |
|2005|95.0     |77.0     |94.0  |71.0      |95.0    |0.0    |0.0        |0.0        |0.0    |
|2006|98.0     |98.0     |97.0  |96.0      |96.0    |0.0    |0.0        |0.0        |0.0    |
|2007|98.0     |99.0     |98.0  |98.0      |96.0    |0.0    |0.0        |0.0        |0.0    |
|2008|96.0     |98.0     |96.0  |98.0      |97.0    |0.0    |0.0        |0.0        |0.0    |
|2009|96.0     |94.0     |95.0  |98.0      |98.0    |0.0    |0.0        |0.0        |0.0    |
|2010|95.0     |98.0     |96.0  |94.0      |96.0    |0.0    |0.0        |0.0        |0.0    |
|2011|96.0     |89.0     |97.0  |95.0      |96.0    |0.0    

+----+---------+---------+------+----------+--------+-------+-----------+-------+
|year|BELISARIO|CARAPUNGO|CENTRO|COTOCOLLAO|EL CAMAL|GUAMANI|LOS CHILLOS|TUMBACO|
+----+---------+---------+------+----------+--------+-------+-----------+-------+
|2004|83.0     |0.0      |87.0  |0.0       |85.0    |0.0    |0.0        |0.0    |
|2005|94.0     |78.0     |92.0  |76.0      |92.0    |65.0   |0.0        |0.0    |
|2006|97.0     |96.0     |95.0  |97.0      |97.0    |96.0   |0.0        |0.0    |
|2007|96.0     |98.0     |96.0  |96.0      |97.0    |98.0   |0.0        |0.0    |
|2008|96.0     |97.0     |96.0  |96.0      |97.0    |98.0   |0.0        |0.0    |
|2009|97.0     |97.0     |95.0  |96.0      |98.0    |96.0   |0.0        |0.0    |
|2010|98.0     |95.0     |96.0  |96.0      |98.0    |96.0   |0.0        |0.0    |
|2011|97.0     |98.0     |98.0  |98.0      |98.0    |98.0   |0.0        |0.0    |
|2012|98.0     |96.0     |99.0  |98.0      |98.0    |98.0   |0.0        |0.0    |
|2013|95.0     |

+----+---------+---------+------+----------+--------+-------+-----------+-----------+-------+
|year|BELISARIO|CARAPUNGO|CENTRO|COTOCOLLAO|EL CAMAL|GUAMANI|LOS CHILLOS|SAN ANTONIO|TUMBACO|
+----+---------+---------+------+----------+--------+-------+-----------+-----------+-------+
|2004|84.0     |0.0      |0.0   |0.0       |94.0    |0.0    |98.0       |0.0        |95.0   |
|2005|93.0     |0.0      |80.0  |78.0      |98.0    |75.0   |98.0       |0.0        |93.0   |
|2006|97.0     |0.0      |94.0  |97.0      |97.0    |98.0   |99.0       |0.0        |95.0   |
|2007|98.0     |49.0     |97.0  |98.0      |97.0    |96.0   |98.0       |0.0        |98.0   |
|2008|96.0     |96.0     |95.0  |97.0      |96.0    |97.0   |96.0       |0.0        |96.0   |
|2009|96.0     |96.0     |96.0  |96.0      |98.0    |96.0   |97.0       |0.0        |97.0   |
|2010|96.0     |97.0     |98.0  |96.0      |97.0    |98.0   |98.0       |0.0        |97.0   |
|2011|96.0     |96.0     |96.0  |97.0      |97.0    |96.0   

In [10]:
first_last = (hist.filter("value is not null")
                  .groupBy("parameter", "station")
                  .agg(F.min("ts_hour").alias("first_value"), F.max("ts_hour").alias("last_value"),
                       F.count("*").alias("values")))
last12m = (hist.filter("ts_hour >= '2025-09-01'")
               .groupBy("parameter", "station")
               .agg(F.round(F.count("value") / F.count("*") * 100, 1).alias("coverage_last_12m_pct")))
first_last.join(last12m, ["parameter", "station"], "left").orderBy("parameter", "station").display(30)

+---------+-----------+-------------------+-------------------+------+---------------------+
|parameter|station    |first_value        |last_value         |values|coverage_last_12m_pct|
+---------+-----------+-------------------+-------------------+------+---------------------+
|NO2_ug   |BELISARIO  |2004-02-09 00:00:00|2026-08-31 23:00:00|189377|92.3                 |
|NO2_ug   |CARAPUNGO  |2005-03-16 17:00:00|2026-08-31 23:00:00|179624|92.0                 |
|NO2_ug   |CENTRO     |2004-01-01 00:00:00|2026-08-31 23:00:00|184854|59.2                 |
|NO2_ug   |COTOCOLLAO |2005-02-25 14:00:00|2026-08-31 23:00:00|161787|87.9                 |
|NO2_ug   |EL CAMAL   |2004-01-01 00:00:00|2026-06-15 13:00:00|152345|73.4                 |
|NO2_ug   |GUAMANI    |2005-03-20 00:00:00|2026-08-30 23:00:00|169702|4.0                  |
|NO2_ug   |LOS CHILLOS|2016-09-15 20:00:00|2026-08-31 23:00:00|79429 |65.5                 |
|NO2_ug   |TUMBACO    |2019-05-29 16:00:00|2026-08-31 23:00:00|59864 |

In [11]:
# Gaps in the last twelve months: runs of consecutive null hours per station and parameter.
w = W.partitionBy("parameter", "station").orderBy("ts_hour")
recent = (hist.filter("ts_hour >= '2025-09-01'")
              .withColumn("is_null", F.col("value").isNull().cast("int"))
              .withColumn("run_id", F.sum((F.col("is_null") != F.lag("is_null", 1, 0).over(w)).cast("int")).over(w)))
runs = (recent.filter("is_null = 1")
              .groupBy("parameter", "station", "run_id")
              .agg(F.count("*").alias("hours"), F.min("ts_hour").alias("gap_start"), F.max("ts_hour").alias("gap_end")))
(runs.groupBy("parameter", "station")
     .agg(F.count("*").alias("gaps"), F.max("hours").alias("longest_gap_hours"),
          F.sum((F.col("hours") > 24).cast("int")).alias("gaps_over_24h"))
     .orderBy("parameter", "station").display(30))
print("Longest gaps:")
runs.orderBy(F.desc("hours")).display(8)

+---------+-----------+----+-----------------+-------------+
|parameter|station    |gaps|longest_gap_hours|gaps_over_24h|
+---------+-----------+----+-----------------+-------------+
|NO2_ug   |BELISARIO  |182 |57               |3            |
|NO2_ug   |CARAPUNGO  |207 |26               |1            |
|NO2_ug   |CENTRO     |193 |2261             |5            |
|NO2_ug   |COTOCOLLAO |185 |241              |3            |
|NO2_ug   |EL CAMAL   |167 |1858             |2            |
|NO2_ug   |GUAMANI    |12  |8200             |2            |
|NO2_ug   |LOS CHILLOS|152 |2380             |3            |
|NO2_ug   |TUMBACO    |168 |77               |1            |
|OZONO_ug |BELISARIO  |221 |197              |6            |
|OZONO_ug |CARAPUNGO  |231 |26               |1            |
|OZONO_ug |CENTRO     |115 |236              |4            |
|OZONO_ug |COTOCOLLAO |194 |51               |2            |
|OZONO_ug |EL CAMAL   |167 |1858             |1            |
|OZONO_ug |GUAMANI    |4

+---------+-----------+------+-----+-------------------+-------------------+
|parameter|station    |run_id|hours|gap_start          |gap_end            |
+---------+-----------+------+-----+-------------------+-------------------+
|PM2.5_ug |GUAMANI    |1     |8293 |2025-09-01 00:00:00|2026-08-12 12:00:00|
|NO2_ug   |GUAMANI    |1     |8200 |2025-09-01 00:00:00|2026-08-08 15:00:00|
|OZONO_ug |GUAMANI    |1     |8200 |2025-09-01 00:00:00|2026-08-08 15:00:00|
|PM2.5_ug |SAN ANTONIO|147   |3275 |2026-04-17 13:00:00|2026-08-31 23:00:00|
|PM2.5_ug |SAN ANTONIO|143   |2520 |2025-11-06 10:00:00|2026-02-19 09:00:00|
|NO2_ug   |LOS CHILLOS|99    |2380 |2025-11-19 12:00:00|2026-02-26 15:00:00|
|NO2_ug   |CENTRO     |79    |2261 |2025-12-02 12:00:00|2026-03-06 16:00:00|
|NO2_ug   |EL CAMAL   |333   |1858 |2026-06-15 14:00:00|2026-08-31 23:00:00|
+---------+-----------+------+-----+-------------------+-------------------+
only showing top 8 rows



> **Finding — nine stations, three generations, and four current outages**
>
> - **2004–2005 founders**: Belisario, Centro, El Camal, Carapungo, Cotocollao (PM2.5 and NO2
>   from the start), plus Guamaní, Los Chillos and Tumbaco for ozone only.
> - **2013–2017 expansion**: PM2.5 reaches Guamaní (2013), Los Chillos (2014), San Antonio and
>   Tumbaco (2017); NO2 reaches Los Chillos (2016) and Tumbaco (2019); San Antonio never
>   measured NO2, which is why the NO2 file has eight columns.
> - Typical coverage for a working station is **93–99 %** of hours per year. Anything far below
>   is an outage, not noise.
>
> Current state, consistent between the export (to 2026-08-31) and the live feed (2026-10-16):
>
> | station | what the history shows | what the feed showed |
> |---|---|---|
> | **El Camal** | all three parameters stop **2026-06-15 13:00**; also dark all of 2015 and 2023–2024 | 0 of 10 layers valid |
> | **Guamaní** | offline **2025-09 → 2026-08-08** (11 months, 8 200 h); back since | valid, 7 of 11 layers |
> | **San Antonio** | PM2.5 stops **2026-04-17**, O3 continuous | no PM2.5 layer at all, 6 layers |
> | **Los Chillos** | NO2 gap 2025-11-19 → 2026-02-26; PM2.5 last value 2026-08-31 13:00 | PM2.5 `IsValid: false`, 9 of 10 valid |
> | Guamaní (NO2) | NO2 last value 2026-08-30 | NO2 `IsValid: false` |
> | Centro | NO2 gap 2025-12-02 → 2026-03-06 | valid |
>
> Consequences for the product: the map must distinguish **"no data for 3 hours"** (stale) from
> **"no data for months"** (station down) from **"never measured here"** (no sensor), and the
> `silver.stations` dimension needs a per-parameter availability window, not a boolean.

---

## 6. The live feed, reconciled with the history

The producer's source is one JSON document with every station. Flatten the captured copy
into rows and check the two things the pipeline depends on: that every feed station maps to a
history column, and that the feed's layers match what the history says each station measures.

In [12]:
probe_dirs = sorted(glob.glob(str(BASE / "probe" / "*" / "secretaria_get_sites_info.json")))
feed_path = Path(probe_dirs[-1])
feed = json.loads(feed_path.read_text(encoding="utf-8"))["d"]
report = json.loads((feed_path.parent / "report.json").read_text(encoding="utf-8"))
fetched_at = next(p for p in report if p["name"] == "get_sites_info")
print("captured:", feed_path.parent.name, "| sites:", len(feed), "| http:", fetched_at["status"], fetched_at["bytes"], "bytes")

_float = lambda x: None if x is None else float(x)   # the feed mixes 0 and 0.0; Spark needs one type
rows = []
for site in feed:
    for layer in site["LayerInfos"]:
        rows.append((site["SourceSiteID"], site["SiteName"], site["AqiTime"], site["AqiValue"], site["AqiCategory"],
                     layer["ParameterName"], layer["UnitName"], _float(layer["Concentration"]), layer["IsValid"],
                     layer["DataTime"], layer["Message"]))
live = spark.createDataFrame(rows, ["station_id", "feed_name", "aqi_time", "iqca", "iqca_category",
                                    "layer", "unit", "value", "is_valid", "data_time", "message"])
dotnet_ms = F.regexp_extract("aqi_time", r"/Date\((-?\d+)", 1).cast("long")
live = (live.withColumn("event_time_utc", F.timestamp_millis(dotnet_ms))
            .withColumn("event_time_local", F.from_utc_timestamp("event_time_utc", QUITO_TZ)))
live.select("feed_name", "event_time_utc", "event_time_local", "iqca", "iqca_category", "layer", "value", "unit", "is_valid", "message").display(100)

captured: 20261016T234146Z | sites: 9 | http: 200 53655 bytes


+----------+-------------------+-------------------+----+-------------+---------+------+-----+--------+-------+
|feed_name |event_time_utc     |event_time_local   |iqca|iqca_category|layer    |value |unit |is_valid|message|
+----------+-------------------+-------------------+----+-------------+---------+------+-----+--------+-------+
|Belisario |2026-10-16 22:00:00|2026-10-16 17:00:00|44  |Deseable     |CO_mg    |0.841 |mg/m3|true    |NULL   |
|Belisario |2026-10-16 22:00:00|2026-10-16 17:00:00|44  |Deseable     |NO2_ug   |50.01 |ug/m3|true    |NULL   |
|Belisario |2026-10-16 22:00:00|2026-10-16 17:00:00|44  |Deseable     |OZONO_ug |28.53 |ug/m3|true    |NULL   |
|Belisario |2026-10-16 22:00:00|2026-10-16 17:00:00|44  |Deseable     |SO2_ug   |2.67  |ug/m3|true    |NULL   |
|Belisario |2026-10-16 22:00:00|2026-10-16 17:00:00|44  |Deseable     |PM2.5_ug |16.56 |ug/m3|true    |NULL   |
|Belisario |2026-10-16 22:00:00|2026-10-16 17:00:00|44  |Deseable     |RAP_VEC  |2.34  |m/s  |true    |N

In [13]:
# Every feed station must have a history column, and vice versa.
joined = stations_df.join(live.select("station_id").distinct(), "station_id", "full")
print("feed stations without reference row:", joined.filter("station is null").count(),
      "| reference rows absent from feed:", joined.filter("station_id is null").count())

# Layers per station in the feed versus parameters with values in the last 12 months of history.
feed_layers = (live.filter(F.col("layer").isin(list(FILES)))
                   .groupBy("station_id").agg(F.collect_set(F.when(F.col("is_valid"), F.col("layer"))).alias("feed_valid_layers"),
                                              F.collect_set("layer").alias("feed_layers")))
hist_recent = (last12m.filter("coverage_last_12m_pct > 0")
                      .groupBy("station").agg(F.collect_set("parameter").alias("history_last_12m")))
(stations_df.join(feed_layers, "station_id", "left").join(hist_recent, "station", "left")
            .select("feed_name", "station", "feed_layers", "feed_valid_layers", "history_last_12m")
            .orderBy("station").display())

feed stations without reference row: 0 | reference rows absent from feed: 0


+----------+-----------+----------------------------+----------------------------+----------------------------+
|feed_name |station    |feed_layers                 |feed_valid_layers           |history_last_12m            |
+----------+-----------+----------------------------+----------------------------+----------------------------+
|Belisario |BELISARIO  |[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug, NO2_ug]|
|Carapungo |CARAPUNGO  |[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug, NO2_ug]|
|Centro    |CENTRO     |[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug, NO2_ug]|
|Cotocollao|COTOCOLLAO |[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug, NO2_ug]|
|El Camal  |EL CAMAL   |[PM2.5_ug, OZONO_ug, NO2_ug]|[]                          |[PM2.5_ug, OZONO_ug, NO2_ug]|
|Guamani   |GUAMANI    |[PM2.5_ug, OZONO_ug, NO2_ug]|[PM2.5_ug, OZONO_ug]        |[PM2.5_ug, OZONO_ug, N

In [14]:
# Publication delay: the feed's event hour versus the moment it was fetched.
from datetime import datetime, timezone
run_started = datetime.strptime(feed_path.parent.name, "%Y%m%dT%H%M%SZ").replace(tzinfo=timezone.utc)
event = live.select(F.max("event_time_utc")).first()[0].replace(tzinfo=timezone.utc)
print(f"event hour {event:%Y-%m-%d %H:%M} UTC | fetched {run_started:%H:%M} UTC | age at fetch: {(run_started - event).total_seconds()/60:.0f} min")

event hour 2026-10-16 22:00 UTC | fetched 23:41 UTC | age at fetch: 102 min


> **Finding — the join works, and the feed is the unvalidated twin of the history**
>
> - All **9 feed stations** map to a history column and back; the GUID ↔ name ↔ column table in
>   `producers/stations.py` is complete. (Jipijapa, listed in the original plan, exists in
>   neither source.)
> - The feed's pollutant layers agree with the history's last twelve months: where a station
>   has no PM2.5 column activity (San Antonio), the feed has no PM2.5 layer; where the history
>   went dark (El Camal), the feed says `IsValid: false`, `Message: "N/A"` on every layer; where
>   a single series stopped at the end of the export (Los Chillos PM2.5, Guamaní NO2), that one
>   layer is invalid and the rest are fine. Three states, all visible in one document.
> - The feed carries **eleven layers**, six pollutants plus five meteorological variables
>   (temperature, solar radiation, rain, wind speed and direction), so REMMAQ itself provides
>   weather at the station; Open-Meteo stays as the gridded complement and forecast.
> - Units differ by pollutant: **CO in mg/m³**, the rest in µg/m³. Silver must carry the unit.
> - The captured hour (17:00 local) was **102 minutes old** when fetched at 18:41 local; the
>   page itself had said "updated 18:09", so the hour-H value appears around **H + 70 min**.
>   Polling every 10 minutes therefore sees a new hour within ≤ 10 min of publication, and a
>   staleness alarm at 3 h (the plan's figure) leaves a safe margin.
> - The feed is **not** the `LIMPIO` series: it is the real-time value before the QA/QC
>   protocol runs. Values the history would later invalidate will appear here first. That is
>   the justification for a quarantine table rather than a filter.

---

## 7. Bronze preview: what the producer writes

If the file sink has been run (`python -m producers.remmaq_producer --sink file`), its JSON
Lines are exactly what Kafka will carry and what Bronze will store. Read them with Spark as
they are, no parsing code: the envelope's `payload` is the site record untouched.

In [15]:
sink_files = glob.glob(str(BASE / "sink" / "aq.measurements.raw" / "*.jsonl"))
if sink_files:
    bronze = spark.read.json(sink_files)
    bronze.printSchema()
    (bronze.select("key", "value.source", "value.schema_version", "value.event_time", "value.fetched_at",
                   "value.payload.SiteName", "value.payload.AqiValue", F.size("value.payload.LayerInfos").alias("layers"))
           .display())
    # The Silver explode, in one expression: one row per station, layer, hour.
    silver_preview = (bronze.select("key", "value.event_time", F.explode("value.payload.LayerInfos").alias("l"))
                            .select("key", "event_time", "l.ParameterName", "l.Concentration", "l.UnitName", "l.IsValid"))
    print("silver rows from", bronze.count(), "bronze records:", silver_preview.count())
    silver_preview.display(12)
else:
    print("no file-sink output under", BASE / "sink", "— run the producer once with SINK=file to populate it")

root
 |-- key: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- event_time: string (nullable = true)
 |    |-- fetched_at: string (nullable = true)
 |    |-- key: string (nullable = true)
 |    |-- payload: struct (nullable = true)
 |    |    |-- AddressFull: string (nullable = true)
 |    |    |-- AqiCategory: string (nullable = true)
 |    |    |-- AqiCategoryClass: string (nullable = true)
 |    |    |-- AqiMessage: string (nullable = true)
 |    |    |-- AqiResponsibleParameterConcentration: double (nullable = true)
 |    |    |-- AqiResponsibleParameterName: string (nullable = true)
 |    |    |-- AqiResponsibleParameterUnitName: string (nullable = true)
 |    |    |-- AqiResponsibleProgramName: string (nullable = true)
 |    |    |-- AqiTime: string (nullable = true)
 |    |    |-- AqiTimeString: string (nullable = true)
 |    |    |-- AqiValue: long (nullable = true)
 |    |    |-- BackgroundColor: string (nullable = tr

+------------------------------------+------+--------------+--------------------+--------------------+----------+--------+------+
|key                                 |source|schema_version|event_time          |fetched_at          |SiteName  |AqiValue|layers|
+------------------------------------+------+--------------+--------------------+--------------------+----------+--------+------+
|f1f5d7e6-09f3-4966-ad15-f6ad47b89973|remmaq|1             |2026-10-16T22:00:00Z|2026-10-17T02:58:11Z|Belisario |44      |10    |
|35e166f8-0d20-4955-84ec-1180d8193d03|remmaq|1             |2026-10-16T22:00:00Z|2026-10-17T02:58:11Z|Carapungo |44      |11    |
|1f090827-f3b4-4c46-9794-af4e2d46c126|remmaq|1             |2026-10-16T22:00:00Z|2026-10-17T02:58:11Z|Centro    |56      |10    |
|da0e2ba0-92ab-4703-80b3-825144c34659|remmaq|1             |2026-10-16T22:00:00Z|2026-10-17T02:58:11Z|Cotocollao|50      |10    |
|f1634ea4-d0d0-4b18-b00a-e25e7c06c8be|remmaq|1             |2026-10-16T22:00:00Z|2026-10-1

silver rows from 9 bronze records: 89
+------------------------------------+--------------------+-------------+-------------+--------+-------+
|key                                 |event_time          |ParameterName|Concentration|UnitName|IsValid|
+------------------------------------+--------------------+-------------+-------------+--------+-------+
|f1f5d7e6-09f3-4966-ad15-f6ad47b89973|2026-10-16T22:00:00Z|CO_mg        |0.841        |mg/m3   |true   |
|f1f5d7e6-09f3-4966-ad15-f6ad47b89973|2026-10-16T22:00:00Z|NO2_ug       |50.01        |ug/m3   |true   |
|f1f5d7e6-09f3-4966-ad15-f6ad47b89973|2026-10-16T22:00:00Z|OZONO_ug     |28.53        |ug/m3   |true   |
|f1f5d7e6-09f3-4966-ad15-f6ad47b89973|2026-10-16T22:00:00Z|SO2_ug       |2.67         |ug/m3   |true   |
|f1f5d7e6-09f3-4966-ad15-f6ad47b89973|2026-10-16T22:00:00Z|PM2.5_ug     |16.56        |ug/m3   |true   |
|f1f5d7e6-09f3-4966-ad15-f6ad47b89973|2026-10-16T22:00:00Z|RAP_VEC      |2.34         |m/s     |true   |
|f1f5d7e6-09f3-49

> **Finding**
>
> One Bronze record per station per poll, keyed by the station GUID, with the site document
> intact. `spark.read.json` infers the nested schema and `explode(LayerInfos)` yields the
> long format Silver needs: **9 records → ~90 rows**. Nothing in Bronze is cast or renamed,
> so if the Secretaría adds a layer tomorrow it lands in Bronze unchanged and only the Silver
> explode needs to know about it.

---

## 8. Value distributions

What does a plausible value look like, per pollutant? Extremes, zeros, negatives and the
shape of the upper tail, across the whole validated history.

In [16]:
values = hist.filter("value is not null")
(values.groupBy("parameter")
       .agg(F.round(F.min("value"), 2).alias("min"),
            F.round(F.expr("percentile(value, 0.005)"), 2).alias("p0_5"),
            F.round(F.expr("percentile(value, 0.5)"), 2).alias("median"),
            F.round(F.avg("value"), 2).alias("mean"),
            F.round(F.expr("percentile(value, 0.995)"), 2).alias("p99_5"),
            F.round(F.max("value"), 2).alias("max"),
            F.sum((F.col("value") < 0).cast("int")).alias("negative"),
            F.sum((F.col("value") == 0).cast("int")).alias("zeros"))
       .orderBy("parameter").display())

+---------+---+----+------+-----+-----+------+--------+-----+
|parameter|min|p0_5|median|mean |p99_5|max   |negative|zeros|
+---------+---+----+------+-----+-----+------+--------+-----+
|NO2_ug   |0.0|1.6 |21.17 |23.17|68.96|123.98|0       |407  |
|OZONO_ug |0.0|0.06|20.79 |25.16|91.49|178.19|0       |3635 |
|PM2.5_ug |0.0|0.0 |14.34 |16.43|62.34|104.7 |0       |7251 |
+---------+---+----+------+-----+-----+------+--------+-----+



In [17]:
# The top of each distribution: are the maxima an episode or a ceiling?
for param, near in [("PM2.5_ug", 100), ("NO2_ug", 120), ("OZONO_ug", 170)]:
    d = values.filter(F.col("parameter") == param)
    print(f"{param}: values > {near}: {d.filter(F.col('value') > near).count()} "
          f"at {d.filter(F.col('value') > near).select('station').distinct().count()} station(s)")
    d.orderBy(F.desc("value")).select("station", "ts_hour", "value").display(3)

PM2.5_ug: values > 100: 87 at 1 station(s)


+--------+-------------------+------+
|station |ts_hour            |value |
+--------+-------------------+------+
|EL CAMAL|2012-10-23 06:00:00|104.7 |
|EL CAMAL|2017-06-29 07:00:00|104.68|
|EL CAMAL|2006-04-20 07:00:00|104.57|
+--------+-------------------+------+
only showing top 3 rows



NO2_ug: values > 120: 13 at 2 station(s)


+--------+-------------------+------+
|station |ts_hour            |value |
+--------+-------------------+------+
|EL CAMAL|2004-10-26 09:00:00|123.98|
|EL CAMAL|2004-01-17 08:00:00|123.73|
|EL CAMAL|2004-02-25 08:00:00|123.72|
+--------+-------------------+------+
only showing top 3 rows



OZONO_ug: values > 170: 5 at 1 station(s)


+-------+-------------------+------+
|station|ts_hour            |value |
+-------+-------------------+------+
|TUMBACO|2004-09-15 11:00:00|178.19|
|TUMBACO|2004-09-14 09:00:00|177.69|
|TUMBACO|2004-09-11 11:00:00|177.14|
+-------+-------------------+------+
only showing top 3 rows



> **Finding — clean below, suspiciously flat on top**
>
> | parameter | median | p99.5 | max | negatives | zeros |
> |---|---|---|---|---|---|
> | PM2.5 | 14.3 | 62.3 | **104.7** | 0 | 7 251 |
> | NO2 | 21.2 | 69.0 | **124.0** | 0 | 407 |
> | O3 | 20.8 | 91.5 | **178.2** | 0 | 3 635 |
>
> - **No negative values** anywhere, and the smallest positive value is 0.01: the validation
>   has already removed instrument negatives, and zeros are most likely below-detection
>   readings clipped to zero (7 251 of them for PM2.5, 0.6 %). Silver should keep zeros but
>   must not treat `0` as "no data".
> - The PM2.5 maximum over 22 years is **104.7 µg/m³**, and the 87 values above 100 all sit at
>   El Camal between 100 and 105. A city with traffic, biomass burning and volcanic ash
>   episodes (Cotopaxi 2015, Sangay 2020) should have seen higher hourly values. The
>   history's upper tail is almost certainly **cut by the validation**, not by the atmosphere.
>
> Consequence: plausibility limits for the live feed must **not** be derived from the history's
> maxima. A live PM2.5 of 150 is rare, but it is exactly the reading the product exists to
> report. The physical limits in section 9 are therefore set by instrument range, and the
> statistical rules flag for review instead of discarding.

---

## 9. The Secretaría's QA/QC rules, applied

`Marco_QAQC_REMMAQ.pdf` describes how `LIMPIO` was produced: physical limits, a global
median/MAD outlier test (`|x − median| > 7·MAD`), distribution tails (p0.5 / p99.5), a spike
check against the previous and next hour (`> 4·MAD` on both sides), and invalidation only when
**two or more** criteria agree. Implementing the same rules in Spark and running them over
`LIMPIO` answers two questions: do they reproduce the Secretaría's decisions (they should,
this is their output), and what do they do to genuine episodes.

In [18]:
# Physical limits are instrument ranges, deliberately wide (section 8). Provisional until the
# Secretaría's own per-analyser limits are obtained.
LIMITS = {"PM2.5_ug": (0.0, 1000.0), "NO2_ug": (0.0, 1000.0), "OZONO_ug": (0.0, 600.0)}
limits = spark.createDataFrame([(k, lo, hi) for k, (lo, hi) in LIMITS.items()], ["parameter", "lo", "hi"])

stats = (values.groupBy("parameter", "station")
               .agg(F.expr("percentile(value, 0.5)").alias("median"),
                    F.expr("percentile(value, 0.005)").alias("p0_5"),
                    F.expr("percentile(value, 0.995)").alias("p99_5")))
mad = (values.join(stats, ["parameter", "station"])
             .withColumn("absdev", F.abs(F.col("value") - F.col("median")))
             .groupBy("parameter", "station").agg(F.expr("percentile(absdev, 0.5)").alias("mad")))

w = W.partitionBy("parameter", "station").orderBy("ts_hour")
flagged = (values.join(stats, ["parameter", "station"]).join(mad, ["parameter", "station"]).join(limits, "parameter")
    .withColumn("f_physical", (F.col("value") < F.col("lo")) | (F.col("value") > F.col("hi")))
    .withColumn("f_mad", F.abs(F.col("value") - F.col("median")) > 7 * F.col("mad"))
    .withColumn("f_tail", (F.col("value") < F.col("p0_5")) | (F.col("value") > F.col("p99_5")))
    .withColumn("prev", F.lag("value").over(w)).withColumn("next", F.lead("value").over(w))
    .withColumn("f_spike", (F.abs(F.col("value") - F.col("prev")) > 4 * F.col("mad")) &
                           (F.abs(F.col("value") - F.col("next")) > 4 * F.col("mad")))
    .withColumn("n_flags", sum(F.col(c).cast("int") for c in ["f_physical", "f_mad", "f_tail", "f_spike"]))
    .withColumn("invalid", F.col("n_flags") >= 2))

(flagged.groupBy("parameter")
        .agg(F.count("*").alias("values"),
             *[F.sum(F.col(c).cast("int")).alias(c) for c in ["f_physical", "f_mad", "f_tail", "f_spike", "invalid"]],
             F.round(F.avg(F.col("invalid").cast("int")) * 100, 2).alias("invalid_pct"))
        .orderBy("parameter").display())

+---------+-------+----------+-----+------+-------+-------+-----------+
|parameter|values |f_physical|f_mad|f_tail|f_spike|invalid|invalid_pct|
+---------+-------+----------+-----+------+-------+-------+-----------+
|NO2_ug   |1176982|0         |1882 |11723 |192    |1914   |0.16       |
|OZONO_ug |1461324|0         |652  |13546 |15     |660    |0.05       |
|PM2.5_ug |1190105|0         |5598 |11117 |4441   |5264   |0.44       |
+---------+-------+----------+-----+------+-------+-------+-----------+



In [19]:
# Per station thresholds the rules imply, and the highest values they would reject.
(stats.join(mad, ["parameter", "station"])
      .select("parameter", "station", F.round("median", 1).alias("median"), F.round("mad", 1).alias("mad"),
              F.round(F.col("median") + 7 * F.col("mad"), 1).alias("median_plus_7mad"), F.round("p99_5", 1).alias("p99_5"))
      .orderBy("parameter", "station").display(30))
flagged.filter("invalid").orderBy(F.desc("value")).select("parameter", "station", "ts_hour", "value", "prev", "next", "f_mad", "f_tail", "f_spike").display(8)

+---------+-----------+------+----+----------------+-----+
|parameter|station    |median|mad |median_plus_7mad|p99_5|
+---------+-----------+------+----+----------------+-----+
|NO2_ug   |BELISARIO  |26.1  |8.6 |86.1            |73.8 |
|NO2_ug   |CARAPUNGO  |15.7  |6.9 |64.2            |61.7 |
|NO2_ug   |CENTRO     |26.9  |9.4 |92.5            |73.2 |
|NO2_ug   |COTOCOLLAO |19.8  |8.6 |80.1            |63.1 |
|NO2_ug   |EL CAMAL   |28.7  |9.2 |93.0            |78.3 |
|NO2_ug   |GUAMANI    |15.6  |7.7 |69.6            |56.0 |
|NO2_ug   |LOS CHILLOS|17.2  |6.8 |65.0            |56.8 |
|NO2_ug   |TUMBACO    |11.8  |5.5 |50.5            |50.6 |
|OZONO_ug |BELISARIO  |18.4  |14.0|116.5           |93.8 |
|OZONO_ug |CARAPUNGO  |22.4  |13.4|115.9           |85.3 |
|OZONO_ug |CENTRO     |17.5  |13.9|114.5           |91.0 |
|OZONO_ug |COTOCOLLAO |19.0  |14.1|117.9           |86.6 |
|OZONO_ug |EL CAMAL   |17.9  |13.7|113.6           |89.5 |
|OZONO_ug |GUAMANI    |26.8  |14.7|129.7           |95.6

+---------+-------+-------------------+------+------+------+-----+------+-------+
|parameter|station|ts_hour            |value |prev  |next  |f_mad|f_tail|f_spike|
+---------+-------+-------------------+------+------+------+-----+------+-------+
|OZONO_ug |TUMBACO|2004-09-15 11:00:00|178.19|174.93|130.81|true |true  |false  |
|OZONO_ug |TUMBACO|2004-09-14 09:00:00|177.69|122.74|160.04|true |true  |false  |
|OZONO_ug |TUMBACO|2004-09-11 11:00:00|177.14|165.31|171.85|true |true  |false  |
|OZONO_ug |TUMBACO|2004-09-15 10:00:00|174.93|121.88|178.19|true |true  |false  |
|OZONO_ug |TUMBACO|2004-09-11 12:00:00|171.85|177.14|151.99|true |true  |false  |
|OZONO_ug |GUAMANI|2005-03-31 14:00:00|169.96|98.4  |58.72 |true |true  |true   |
|OZONO_ug |TUMBACO|2004-11-12 12:00:00|168.77|161.66|99.6  |true |true  |false  |
|OZONO_ug |GUAMANI|2005-03-30 13:00:00|166.8 |162.38|149.97|true |true  |false  |
+---------+-------+-------------------+------+------+------+-----+------+-------+
only showing top

> **Finding — the rules agree with the history, and they bite real episodes**
>
> Re-running the protocol over its own output re-flags **0.05 % (O3), 0.16 % (NO2) and
> 0.44 % (PM2.5)** of values. The rules reproduce the Secretaría's decisions to within half a
> percent, which validates the implementation. The residue is instructive:
>
> - The highest rejected values are **consecutive hours** of ozone at Tumbaco in September 2004
>   (165 → 177 → 178 → 175 µg/m³) and at Guamaní in March 2005. Those are episodes, not
>   spikes: the spike rule correctly does *not* fire, but the global MAD and tail rules do,
>   together, and the two-criteria vote invalidates them.
> - A threshold of `median + 7·MAD` is about **60 µg/m³ for PM2.5** and **115 for O3**. Both are
>   values the city really reaches on a bad day. A global MAD computed over 22 years is a
>   measure of *typical* variability; by construction it calls every atypical day an outlier.
>
> Rules for Silver, derived from this:
>
> 1. **Quarantine automatically** on the physical limits (negative, above instrument range) and
>    on the spike rule (single-hour jump with no neighbour support). These catch instrument
>    faults and cannot hide a real episode, since an episode has neighbours.
> 2. **Flag, do not quarantine**, on MAD and tails. Keep the value in `silver.measurements`
>    with a `review` flag, so that the map shows the bad day and the analyst sees the flag.
> 3. Compute MAD per **station, parameter and month of year** rather than globally, once
>    enough live data exists; a September ozone value should be judged against Septembers.
> 4. Record every rule decision in `silver.measurements_quarantine` with the rule name, the
>    threshold and the value, never a bare "invalid".

---

## 10. The IQCA, reverse-engineered from the feed

The feed reports the Índice Quiteño de Calidad del Aire (IQCA) per station, and for each
pollutant the index value and the concentration it was computed from. Two things can be
read off it without documentation: the index scale for low concentrations, and the averaging
window, because the concentration the index uses is not the hourly value in `LayerInfos`.

In [20]:
pairs = []
for site in feed:
    hourly = {l["ParameterName"]: l["Concentration"] for l in site["LayerInfos"]}
    for r in site["SiteAqiResults"]:
        p = r["AqiResponsibleParameterName"]
        pairs.append((p, site["SiteName"], _float(hourly.get(p)), _float(r["AqiResponsibleParameterConcentration"]), r["AqiValue"], r["AqiCategory"]))
iqca = spark.createDataFrame(pairs, ["layer", "station", "hourly_value", "index_concentration", "iqca", "category"])
iqca = iqca.withColumn("index_per_unit", F.round(F.col("iqca") / F.col("index_concentration"), 2))
iqca.orderBy("layer", "index_concentration").display(40)
(iqca.groupBy("layer").agg(F.count("*").alias("points"), F.round(F.avg("index_per_unit"), 2).alias("slope"),
                           F.max("index_concentration").alias("max_conc_seen"), F.max("iqca").alias("max_iqca_seen"))
     .orderBy("layer").display())

+--------+----------+------------+-------------------+----+---------+--------------+
|layer   |station   |hourly_value|index_concentration|iqca|category |index_per_unit|
+--------+----------+------------+-------------------+----+---------+--------------+
|CO_mg   |Cotocollao|0.042       |0.028              |0   |Deseable |0.0           |
|CO_mg   |Carapungo |0.506       |0.388              |4   |Deseable |10.31         |
|CO_mg   |Centro    |0.416       |0.391              |4   |Deseable |10.23         |
|CO_mg   |Belisario |0.841       |0.623              |6   |Deseable |9.63          |
|CO_mg   |LosChillos|0.74        |0.767              |8   |Deseable |10.43         |
|NO2_ug  |Carapungo |16.64       |16.64              |8   |Deseable |0.48          |
|NO2_ug  |Cotocollao|22.22       |22.22              |11  |Deseable |0.5           |
|NO2_ug  |Tumbaco   |22.4        |22.4               |11  |Deseable |0.49          |
|NO2_ug  |LosChillos|24.25       |24.25              |12  |Deseab

+--------+------+-----+-------------+-------------+
|layer   |points|slope|max_conc_seen|max_iqca_seen|
+--------+------+-----+-------------+-------------+
|CO_mg   |5     |8.12 |0.767        |8            |
|NO2_ug  |6     |0.49 |50.01        |25           |
|OZONO_ug|8     |1.01 |55.7         |56           |
|PM10_ug |3     |1.0  |39.0         |39           |
|PM2.5_ug|6     |2.0  |22.2         |44           |
|SO2_ug  |7     |0.78 |14.5         |12           |
+--------+------+-----+-------------+-------------+



> **Finding — first segment of the scale, and the averaging windows**
>
> | pollutant | index per unit | implied first breakpoint (index 50) | hourly ≠ index concentration? | likely window |
> |---|---|---|---|---|
> | PM2.5 | **2.0** | 25 µg/m³ | yes (e.g. 16.6 hourly vs 18.1) | 24 h mean |
> | PM10 | 1.0 | 50 µg/m³ | yes | 24 h mean |
> | O3 | 1.0 | 50 µg/m³ | yes (28.5 vs 44.3) | 8 h mean |
> | NO2 | 0.5 | 100 µg/m³ | **no** (50.01 = 50.01) | 1 h |
> | SO2 | ≈ 0.83 | 60 µg/m³ | yes | 24 h mean |
> | CO | 10 | 5 mg/m³ | yes | 8 h mean |
>
> These slopes hold only for the range observed on one afternoon (index ≤ 56); the scale is
> piecewise linear and the upper segments (Precaución, Alerta, Alarma, Emergencia) are not
> observable from a clean day. The station IQCA is the **maximum** over pollutants (Belisario:
> 44 from O3, 36 from PM2.5 → 44), and the feed names the responsible pollutant.
>
> Consequences: Gold can reuse the feed's IQCA for "now" without computing anything, but the
> trend charts and the alert thresholds over history need the full breakpoint table and the
> rolling windows (24 h for PM2.5, 8 h for O3 and CO). The table must come from the
> Secretaría's IQCA definition (Ordenanza Metropolitana), an input to obtain before Gold.

---

## 11. What the data can show

Three views the product promises, checked against the history to see that the signal is
there: the long-term trend per station, the daily cycle, and "how often is it unhealthy".
For the last one the 24-hour PM2.5 mean is compared with the WHO 2021 guideline
(15 µg/m³) and Ecuador's NECA daily limit (50 µg/m³), two published thresholds that do not
depend on the IQCA table still to be obtained.

In [21]:
print("PM2.5 yearly mean per station (µg/m³)")
(hist.filter("parameter = 'PM2.5_ug'")
     .groupBy(F.year("ts_hour").alias("year")).pivot("station").agg(F.round(F.avg("value"), 1))
     .orderBy("year").display(30))

PM2.5 yearly mean per station (µg/m³)


+----+---------+---------+------+----------+--------+-------+-----------+-----------+-------+
|year|BELISARIO|CARAPUNGO|CENTRO|COTOCOLLAO|EL CAMAL|GUAMANI|LOS CHILLOS|SAN ANTONIO|TUMBACO|
+----+---------+---------+------+----------+--------+-------+-----------+-----------+-------+
|2004|19.6     |NULL     |16.4  |NULL      |22.4    |NULL   |NULL       |NULL       |NULL   |
|2005|21.3     |21.1     |20.8  |19.5      |26.4    |NULL   |NULL       |NULL       |NULL   |
|2006|21.5     |17.3     |20.8  |18.5      |24.4    |NULL   |NULL       |NULL       |NULL   |
|2007|21.8     |19.0     |21.0  |16.5      |23.6    |NULL   |NULL       |NULL       |NULL   |
|2008|17.2     |14.4     |19.1  |14.3      |20.6    |NULL   |NULL       |NULL       |NULL   |
|2009|18.0     |19.4     |18.4  |16.9      |22.3    |NULL   |NULL       |NULL       |NULL   |
|2010|18.2     |18.5     |19.1  |16.2      |23.0    |NULL   |NULL       |NULL       |NULL   |
|2011|17.0     |17.2     |18.1  |14.7      |21.5    |NULL   

In [22]:
# 24 h rolling mean of PM2.5 (needs at least 18 of 24 hours), then the share of days over each threshold.
w24 = W.partitionBy("station").orderBy(F.unix_timestamp("ts_hour")).rangeBetween(-23 * 3600, 0)
pm = (hist.filter("parameter = 'PM2.5_ug'")
          .withColumn("mean_24h", F.avg("value").over(w24))
          .withColumn("n_24h", F.count("value").over(w24))
          .filter("n_24h >= 18 and hour(ts_hour) = 23"))           # one value per day, the day's closing 24 h mean
(pm.groupBy(F.year("ts_hour").alias("year"))
   .agg(F.count("*").alias("station_days"),
        F.round(F.avg((F.col("mean_24h") > 15).cast("int")) * 100, 1).alias("pct_days_over_who_15"),
        F.round(F.avg((F.col("mean_24h") > 50).cast("int")) * 100, 2).alias("pct_days_over_neca_50"))
   .filter("year >= 2017").orderBy("year").display())
print("last 12 months, per station:")
(pm.filter("ts_hour >= '2025-09-01'").groupBy("station")
   .agg(F.count("*").alias("days"), F.round(F.avg("mean_24h"), 1).alias("mean_24h_avg"),
        F.round(F.avg((F.col("mean_24h") > 15).cast("int")) * 100, 1).alias("pct_days_over_who_15"),
        F.round(F.avg((F.col("mean_24h") > 50).cast("int")) * 100, 2).alias("pct_days_over_neca_50"))
   .orderBy("station").display())

+----+------------+--------------------+---------------------+
|year|station_days|pct_days_over_who_15|pct_days_over_neca_50|
+----+------------+--------------------+---------------------+
|2017|2889        |61.3                |0.0                  |
|2018|3155        |59.0                |0.0                  |
|2019|3085        |48.4                |0.0                  |
|2020|2946        |32.0                |0.0                  |
|2021|3027        |35.0                |0.0                  |
|2022|3132        |50.5                |0.0                  |
|2023|2709        |38.7                |0.0                  |
|2024|2420        |41.4                |0.0                  |
|2025|2782        |30.1                |0.0                  |
|2026|1573        |21.7                |0.0                  |
+----+------------+--------------------+---------------------+

last 12 months, per station:


+-----------+----+------------+--------------------+---------------------+
|station    |days|mean_24h_avg|pct_days_over_who_15|pct_days_over_neca_50|
+-----------+----+------------+--------------------+---------------------+
|BELISARIO  |351 |14.3        |41.0                |0.0                  |
|CARAPUNGO  |352 |14.2        |38.9                |0.0                  |
|CENTRO     |350 |12.7        |26.3                |0.0                  |
|COTOCOLLAO |354 |12.6        |21.5                |0.0                  |
|EL CAMAL   |283 |15.4        |51.2                |0.0                  |
|GUAMANI    |15  |11.1        |6.7                 |0.0                  |
|LOS CHILLOS|295 |12.2        |21.4                |0.0                  |
|SAN ANTONIO|33  |15.1        |45.5                |0.0                  |
|TUMBACO    |347 |9.1         |3.7                 |0.0                  |
+-----------+----+------------+--------------------+---------------------+



> **Finding — the signal is there, and it is a story about ordinary days, not disasters**
>
> - **Long-term trend**: PM2.5 yearly means fell from ~20 µg/m³ (2005–2010) to ~12–15 (2020s)
>   at the founding stations. The **2020 lockdown dip** is visible at every station
>   (Belisario 14.9 → 12.4, Centro 15.4 → 12.7) with a rebound in 2022. El Camal is consistently
>   the most polluted site; Tumbaco, in the valley, the cleanest (8–10 in recent years).
> - **Daily cycle**: the 07:00 PM2.5 and NO2 peak and the noon ozone peak (section 4) are strong
>   enough that "is it a good time to run" has a real, hour-dependent answer.
> - **How often unhealthy**: in 2025 the 24 h PM2.5 mean exceeded the WHO guideline on
>   **30 % of station-days** (from 4 % at Tumbaco to 51 % at El Camal over the last twelve
>   months) but exceeded Ecuador's 50 µg/m³ limit on **none**. Under the national scale nearly
>   every day is "Deseable"; under WHO guidance a large share is not. The product should show
>   both, with the source of each, and let alert thresholds be chosen on either scale.

---

## Summary

### Established

- **Source of truth**: REMMAQ, nine stations, live through the Secretaría's Envista portal
  (undocumented page method, one POST, every station and layer), history through validated
  xlsx exports back to **2004**. OpenAQ is stale since 2021 and out of the design.
- History is **local time (UTC-5)** with a drifting export clock since 2025-12: round to the
  nearest hour, convert with `America/Guayaquil`. The live feed is UTC epoch. Silver key:
  **(station, parameter, hour in UTC)**.
- Three station identifiers (feed GUID, feed name, xlsx header) map one-to-one; the table in
  `producers/stations.py` is complete for all nine.
- Coverage for a working station is 93–99 % per year. Four current outages are known and
  consistent across both sources (El Camal since 2026-06-15, Guamaní Sep 2025 → Aug 2026,
  San Antonio PM2.5 since 2026-04-17, two NO2 gaps over the winter).
- `LIMPIO` contains no negatives and an upper tail that stops abruptly near 105 µg/m³
  (PM2.5); the live feed is the unvalidated series and will exceed that.
- The Secretaría's QA/QC rules are reproducible in Spark to within 0.5 %, and the global MAD
  and tail rules flag genuine multi-hour episodes. Silver quarantines on physical limits and
  spikes, flags on MAD and tails, and records every decision.
- Publication delay of the hourly value: about **70 minutes**; a 10-minute poll and a 3-hour
  staleness threshold are appropriate.
- The feed's IQCA is directly usable for "now"; the full breakpoint table and the rolling
  windows (24 h PM2.5, 8 h O3/CO) are needed for history and alerts.

### Known limitations

- Only three of six pollutants were downloaded (PM2.5, NO2, O3). CO, SO2 and PM10 exports
  exist on the portal and should be pulled before the backfill.
- Hour label convention (ending vs starting) unconfirmed; irrelevant at hourly grain.
- The history's upper tail is shaped by validation; extreme-event statistics from it are
  biased low.
- The live endpoint is an internal page method of a vendor product. It can change without
  notice; the producer validates the shape and fails loudly, and the data contract records the risk.
- The 2026-09-01 → first Bronze record window is in neither source until the next export.

### Objectives for Stage 1, stated as things the data supports

1. A map of **nine stations** with the current IQCA and hourly values, distinguishing stale,
   down and not-measured-here. *Supported by the live feed as captured.*
2. 24 h and 7 d trends per station and parameter. *Supported once Bronze accumulates a week;
   backfillable to 2004 from the exports.*
3. A plain "is it a good time to exercise" answer using the hour of day, the 24 h PM2.5 mean
   and the station IQCA. *Supported; thresholds on WHO and national scales.*
4. Alerts on threshold crossings per station and parameter, with hysteresis. *Supported in
   principle; needs the IQCA table for index-based thresholds, concentration thresholds work today.*
5. A pipeline health view: publication delay per poll, stations offline, quarantine rate.
   *Supported; every input is already in the producer summary and the Silver rules.*